In [1]:
!pip install -q scikit-learn
!pip install -q xgboost

In [2]:
import pandas as pd
import numpy as np
import time

from scipy.stats import t
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.model_selection import RepeatedStratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef
)

import sklearn
import xgboost

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)

scikit-learn: 1.6.1
xgboost: 3.4.1


In [3]:
SEED = 42

df = pd.read_csv(
    '/kaggle/input/datasets/aderahmad/stroke-dataset-raw/healthcare-dataset-stroke-data.csv'
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (5110, 12)
      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  32.5     never smoked   
3        Private          Urban             171.23  34.4           smokes   
4  Self-employed          Rural             174.12  24.0     never smoked   

   stroke  
0       1  
1       1  
2       1  
3       1  
4       1  


In [4]:
X = df.drop(columns=['stroke'])
y = df['stroke']

# ID tidak digunakan sebagai feature
X = X.drop(columns=['id'])

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

print("\nClass proportion:")
print(y.value_counts(normalize=True))

X shape: (5110, 10)
y shape: (5110,)

Class distribution:
stroke
0    4861
1     249
Name: count, dtype: int64

Class proportion:
stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64


In [5]:
numeric_features = [
    'age',
    'hypertension',
    'heart_disease',
    'avg_glucose_level',
    'bmi'
]

onehot_features = [
    'gender',
    'work_type',
    'smoking_status'
]

binary_features = [
    'ever_married',
    'Residence_type'
]

In [6]:
binary_categories = [
    ['No', 'Yes'],
    ['Rural', 'Urban']
]

In [7]:
numeric_transformer_tree = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean'))
    ]
)

onehot_transformer_tree = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ))
    ]
)

binary_transformer_tree = Pipeline(
    steps=[
        ('encoder', OrdinalEncoder(
            categories=binary_categories,
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_tree, numeric_features),
        ('onehot', onehot_transformer_tree, onehot_features),
        ('binary', binary_transformer_tree, binary_features)
    ]
)

In [8]:
numeric_transformer_linear = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

onehot_transformer_linear = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ))
    ]
)

binary_transformer_linear = Pipeline(
    steps=[
        ('encoder', OrdinalEncoder(
            categories=binary_categories,
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ]
)

preprocessor_linear = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_linear, numeric_features),
        ('onehot', onehot_transformer_linear, onehot_features),
        ('binary', binary_transformer_linear, binary_features)
    ]
)

In [9]:
models = {

    'Random Forest': Pipeline(
        steps=[
            ('preprocessor', preprocessor_tree),
            ('classifier', RandomForestClassifier(
                random_state=SEED
            ))
        ]
    ),

    'XGBoost': Pipeline(
        steps=[
            ('preprocessor', preprocessor_tree),
            ('classifier', XGBClassifier(
                random_state=SEED,
                eval_metric='logloss'
            ))
        ]
    ),

    'Logistic Regression': Pipeline(
        steps=[
            ('preprocessor', preprocessor_linear),
            ('classifier', LogisticRegression(
                random_state=SEED,
                max_iter=1000
            ))
        ]
    ),

    'SVM': Pipeline(
        steps=[
            ('preprocessor', preprocessor_linear),
            ('classifier', SVC(
                probability=True,
                random_state=SEED
            ))
        ]
    )
}

In [10]:
cv = RepeatedStratifiedKFold(
    n_splits=10,
    n_repeats=2,
    random_state=SEED
)

# Precompute once so every model uses exactly the same train/test folds.
cv_splits = list(cv.split(X, y))
print(f"Total CV splits per model: {len(cv_splits)}")

Total CV splits per model: 20


In [11]:
fold_results = []

summary_results = []

In [12]:
for model_name, model in models.items():

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    for fold_number, (train_idx, test_idx) in enumerate(
        cv_splits,
        start=1
    ):

        # =========================================
        # SPLIT OUTER FOLD
        # =========================================

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # =========================================
        # FIT MODEL
        # =========================================

        start_time = time.perf_counter()
        model.fit(X_train, y_train)
        runtime_seconds = time.perf_counter() - start_time

        # =========================================
        # PREDICTION
        # =========================================

        y_pred = model.predict(X_test)

        y_prob = model.predict_proba(X_test)[:, 1]

        # =========================================
        # CONFUSION MATRIX
        # =========================================

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1]
        ).ravel()

        # =========================================
        # METRICS
        # =========================================

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        average_precision = average_precision_score(
            y_test,
            y_prob
        )

        # =========================================
        # PR-AUC
        # =========================================

        precision_curve, recall_curve, _ = precision_recall_curve(
            y_test,
            y_prob
        )

        pr_auc = auc(
            recall_curve,
            precision_curve
        )

        # =========================================
        # SPECIFICITY
        # =========================================

        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        # =========================================
        # BALANCED ACCURACY
        # =========================================

        balanced_accuracy = balanced_accuracy_score(
            y_test,
            y_pred
        )

        # =========================================
        # MCC
        # =========================================

        mcc = matthews_corrcoef(
            y_test,
            y_pred
        )

        # =========================================
        # REPEAT / FOLD
        # =========================================

        repeat_number = ((fold_number - 1) // 10) + 1
        fold_in_repeat = ((fold_number - 1) % 10) + 1

        # =========================================
        # SAVE RESULT
        # =========================================

        fold_results.append({

            'Model': model_name,
            'Repeat': repeat_number,
            'Fold': fold_in_repeat,
            'Runtime_Seconds': runtime_seconds,

            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'Specificity': specificity,
            'F1': f1,

            'ROC_AUC': roc_auc,
            'Average_Precision': average_precision,
            'PR_AUC': pr_auc,

            'Balanced_Accuracy': balanced_accuracy,
            'MCC': mcc,

            'TN': tn,
            'FP': fp,
            'FN': fn,
            'TP': tp

        })

        print(
            f"Repeat {repeat_number} | "
            f"Fold {fold_in_repeat} | "
            f"BA={balanced_accuracy:.4f} | "
            f"MCC={mcc:.4f} | "
            f"PR-AUC={pr_auc:.4f} | "
            f"Runtime={runtime_seconds:.2f}s"
        )


MODEL: Decision Tree
Repeat 1 | Fold 1 | BA=0.6763 | MCC=0.3095 | PR-AUC=0.3662 | Runtime=0.05s
Repeat 1 | Fold 2 | BA=0.4994 | MCC=-0.0013 | PR-AUC=0.0673 | Runtime=0.04s
Repeat 1 | Fold 3 | BA=0.5533 | MCC=0.0977 | PR-AUC=0.1672 | Runtime=0.03s
Repeat 1 | Fold 4 | BA=0.5533 | MCC=0.0977 | PR-AUC=0.1672 | Runtime=0.04s
Repeat 1 | Fold 5 | BA=0.5153 | MCC=0.0301 | PR-AUC=0.1010 | Runtime=0.03s
Repeat 1 | Fold 6 | BA=0.5835 | MCC=0.1816 | PR-AUC=0.2386 | Runtime=0.03s
Repeat 1 | Fold 7 | BA=0.5133 | MCC=0.0251 | PR-AUC=0.0982 | Runtime=0.04s
Repeat 1 | Fold 8 | BA=0.5374 | MCC=0.0747 | PR-AUC=0.1415 | Runtime=0.04s
Repeat 1 | Fold 9 | BA=0.5615 | MCC=0.1307 | PR-AUC=0.1915 | Runtime=0.03s
Repeat 1 | Fold 10 | BA=0.4962 | MCC=-0.0075 | PR-AUC=0.0633 | Runtime=0.03s
Repeat 2 | Fold 1 | BA=0.5384 | MCC=0.0783 | PR-AUC=0.1440 | Runtime=0.03s
Repeat 2 | Fold 2 | BA=0.5702 | MCC=0.1215 | PR-AUC=0.1931 | Runtime=0.03s
Repeat 2 | Fold 3 | BA=0.5184 | MCC=0.0383 | PR-AUC=0.1060 | Runtime=0.03s


In [13]:
fold_df = pd.DataFrame(fold_results)

print("\nShape fold results:")
print(fold_df.shape)

fold_df.head()


Shape fold results:
(100, 18)


,Model,Repeat,Fold,Runtime_Seconds,Accuracy,Precision,Recall,Specificity,F1,ROC_AUC,Average_Precision,PR_AUC,Balanced_Accuracy,MCC,TN,FP,FN,TP
0,Decision Tree,1,1,0.048259,0.925636,0.303030,0.40,0.952675,0.344828,0.676337,0.150566,0.366192,0.676337,0.309522,463,23,15,10
1,Decision Tree,1,2,0.038034,0.913894,0.047619,0.04,0.958848,0.043478,0.499424,0.048871,0.067293,0.499424,-0.001252,466,20,24,1
2,Decision Tree,1,3,0.034820,0.908023,0.133333,0.16,0.946502,0.145455,0.553251,0.062429,0.167215,0.553251,0.097727,460,26,21,4
3,Decision Tree,1,4,0.036921,0.908023,0.133333,0.16,0.946502,0.145455,0.553251,0.062429,0.167215,0.553251,0.097727,460,26,21,4
4,Decision Tree,1,5,0.033034,0.908023,0.076923,0.08,0.950617,0.078431,0.515309,0.051164,0.100966,0.515309,0.030054,462,24,23,2


In [14]:
runtime_summary = (
    fold_df
    .groupby('Model')['Runtime_Seconds']
    .agg(['mean', 'std', 'sum'])
)

runtime_summary

,mean,std,sum
Model,,,
Decision Tree,0.035619,0.003360,0.712370
Logistic Regression,0.050949,0.002787,1.018980
Random Forest,0.527205,0.011942,10.544103
SVM,1.610101,0.040246,32.202018
XGBoost,0.111050,0.020990,2.220996


In [15]:
metrics = [
    'Accuracy',
    'Precision',
    'Recall',
    'Specificity',
    'F1',
    'ROC_AUC',
    'Average_Precision',
    'PR_AUC',
    'Balanced_Accuracy',
    'MCC'
]

summary_df = (
    fold_df
    .groupby('Model')[metrics]
    .agg(['mean', 'std'])
)

# Calculate 95% Confidence Interval
ci_results = {}

for model_name, model_data in fold_df.groupby('Model'):

    ci_results[model_name] = {}

    n = len(model_data)
    df_ci = n - 1
    t_critical = t.ppf(0.975, df_ci)

    for metric in metrics:

        mean = model_data[metric].mean()
        std = model_data[metric].std()
        margin_error = t_critical * (std / np.sqrt(n))

        ci_lower = mean - margin_error
        ci_upper = mean + margin_error

        ci_results[model_name][metric] = (
            ci_lower,
            ci_upper
        )

# Display 95% CI
for model_name in ci_results:

    print(f"\n{model_name}")

    for metric in metrics:

        lower, upper = ci_results[model_name][metric]

        print(
            f"{metric}: "
            f"{lower:.4f} - {upper:.4f}"
        )

summary_df


Decision Tree
Accuracy: 0.9068 - 0.9155
Precision: 0.1074 - 0.1684
Recall: 0.1198 - 0.2093
Specificity: 0.9442 - 0.9546
F1: 0.1130 - 0.1832
ROC_AUC: 0.5357 - 0.5783
Average_Precision: 0.0575 - 0.0798
PR_AUC: 0.1361 - 0.2072
Balanced_Accuracy: 0.5357 - 0.5783
MCC: 0.0669 - 0.1398

Logistic Regression
Accuracy: 0.9511 - 0.9518
Precision: -0.0441 - 0.2441
Recall: -0.0018 - 0.0098
Specificity: 1.0000 - 1.0000
F1: -0.0034 - 0.0188
ROC_AUC: 0.8277 - 0.8516
Average_Precision: 0.1933 - 0.2406
PR_AUC: 0.1785 - 0.2264
Balanced_Accuracy: 0.4991 - 0.5049
MCC: -0.0086 - 0.0476

Random Forest
Accuracy: 0.9488 - 0.9506
Precision: 0.0119 - 0.2714
Recall: 0.0017 - 0.0183
Specificity: 0.9969 - 0.9988
F1: 0.0031 - 0.0339
ROC_AUC: 0.7909 - 0.8135
Average_Precision: 0.1456 - 0.1846
PR_AUC: 0.1357 - 0.1743
Balanced_Accuracy: 0.4997 - 0.5081
MCC: -0.0033 - 0.0596

SVM
Accuracy: 0.9510 - 0.9516
Precision: 0.0000 - 0.0000
Recall: 0.0000 - 0.0000
Specificity: 1.0000 - 1.0000
F1: 0.0000 - 0.0000
ROC_AUC: 0.5921

Accuracy           Precision              Recall  \
                         mean       std      mean       std      mean   
Model                                                                   
Decision Tree        0.911155  0.009275  0.137924  0.065114  0.164583   
Logistic Regression  0.951468  0.000803  0.100000  0.307794  0.004000   
Random Forest        0.949706  0.001915  0.141667  0.277178  0.010000   
SVM                  0.951272  0.000602  0.000000  0.000000  0.000000   
XGBoost              0.942955  0.004805  0.229583  0.157978  0.068250   

                              Specificity                  F1            \
                          std        mean       std      mean       std   
Model                                                                     
Decision Tree        0.095610    0.949395  0.011128  0.148110  0.075009   
Logistic Regression  0.012312    1.000000  0.000000  0.007692  0.023676   
Random Forest        0.017770    0.997841  0.002052  0.018529  0.032938   
SVM                  0.000000    1.000000  0.000000  0.000000  0.000000   
XGBoost              0.043237    0.987760  0.004925  0.102454  0.061968   

                      ROC_AUC           Average_Precision              PR_AUC  \
                         mean       std              mean       std      mean   
Model                                                                           
Decision Tree        0.556989  0.045517          0.068668  0.023818  0.171606   
Logistic Regression  0.839640  0.025567          0.216925  0.050504  0.202477   
Random Forest        0.802212  0.024147          0.165129  0.041628  0.154960   
SVM                  0.613283  0.045277          0.122947  0.038356  0.112256   
XGBoost              0.807272  0.026376          0.179655  0.040055  0.166269   

                              Balanced_Accuracy                 MCC            
                          std              mean       std      mean       std  
Model                                                                          
Decision Tree        0.075953          0.556989  0.045517  0.103351  0.077857  
Logistic Regression  0.051205          0.502000  0.006156  0.019524  0.060093  
Random Forest        0.041221          0.503920  0.008974  0.028124  0.067163  
SVM                  0.039801          0.500000  0.000000  0.000000  0.000000  
XGBoost              0.039361          0.528005  0.021466  0.099271  0.074770

In [16]:
confusion_summary = (
    fold_df
    .groupby('Model')[['TN', 'FP', 'FN', 'TP']]
    .sum()
)

confusion_summary

,TN,FP,FN,TP
Model,,,,
Decision Tree,9230,492,416,82
Logistic Regression,9722,0,496,2
Random Forest,9701,21,493,5
SVM,9722,0,498,0
XGBoost,9603,119,464,34


In [17]:
fold_df.to_csv(
    'baseline_repeated_fold_results.csv',
    index=False
)

summary_long = []

for model_name in fold_df['Model'].unique():

    model_data = fold_df[
        fold_df['Model'] == model_name
    ]

    n = len(model_data)
    df_ci = n - 1
    t_critical = t.ppf(0.975, df_ci)

    for metric in metrics:

        mean = model_data[metric].mean()
        std = model_data[metric].std()

        margin_error = (
            t_critical *
            (std / np.sqrt(n))
        )

        ci_lower = mean - margin_error
        ci_upper = mean + margin_error

        summary_long.append({

            'Model': model_name,
            'Metric': metric,
            'Mean': mean,
            'Std': std,
            'CI_95_Lower': ci_lower,
            'CI_95_Upper': ci_upper

        })

summary_long_df = pd.DataFrame(summary_long)

summary_long_df.to_csv(
    'baseline_results.csv',
    index=False
)

confusion_summary.to_csv(
    'baseline_confusion_matrix.csv'
)

runtime_summary.to_csv(
    'baseline_runtime.csv'
)

print("All baseline results saved.")

All baseline results saved.
